# Практическое задание №1. Фурье-анализ сигналов, дискретизация, алиасинг

## 1. Работа с GitHub и Google Colab

### Форк учебного репозитория
1. Зайдите на страницу репозитория курса (например, `https://github.com/korzhimanov/dsp-seminars`).
2. Нажмите кнопку **Fork** (справа вверху) – создастся ваша копия репозитория.

### Создание своей ветки для редактирования
1. Зайдите на страницу скопированного к себе репозитория.
2. Нажмите кнопку **Branches** (над списком файлов) и затем **New branch** (слева вверху), назовите ветку своей фамилией.

### Открытие ноутбука в Colab
- Перейдите в свой форк, смените ветку на вашу, откройте нужный `.ipynb` файл.
- В адресной строке замените `github.com` на `colab.research.google.com/github/` – откроется ноутбук в Colab.
- Подключите свой GitHub к Colab (если ещё нет)
- Либо используйте прямой значок "Open in Colab" (если есть).
- Либо зайдите в Colab, нажмите "Открыть ноутбук" в меню (`Ctrl+O`), найдите нужный репоизторий, смените ветку на вашу и откройте нужный файл

### Сохранение результатов и создание Pull Request (PR)
После выполнения задания нужно сохранить изменения в своём форке и отправить PR в основной репозиторий.

В Colab сохраните (`Ctrl+S`) ноутбук и заполните описание коммита.

Зайдите на GitHub и создайте Pull Request из вашего форка в основной репозиторий (вкладка "Pull requests", кнопка "New pull request").

## 2. Основные функции NumPy для создания векторов

- **`np.linspace(start, stop, num)`** – равномерно распределённые точки между start и stop (включая stop).
  ```python
  t = np.linspace(0, 1, 1000, endpoint=False)  # 1000 точек от 0 до 1
  ```
- **`np.arange(start, stop, step)`** – точки с фиксированным шагом (stop не включается).
  ```python
  t = np.arange(0, 1, 0.001)   # шаг 0.001 с
  ```

## 3. Генерация сигналов

### Синусоида
$$
x(t) = A \sin(\omega t + \phi) = A \sin(2\pi f t + \phi)
$$
```python
f = 5          # частота, Гц
A = 1          # амплитуда
phi = 0        # начальная фаза (рад)
x = A * np.sin(2 * np.pi * f * t + phi)
```

### Меандр (прямоугольный сигнал)
Функция `signal.square` из scipy.signal:
```python
f_meandr = 2
meandr = signal.square(2 * np.pi * f_meandr * t)
# Можно задать скважность параметром duty (по умолчанию 0.5 – 50%)
```

## 4. Построение графиков (matplotlib)

### Основные команды
- **`plt.plot(x, y)`** – линейный график.
- **`plt.stem(x, y)`** – дискретные отсчёты ("палочки").
- **`plt.xlabel(), plt.ylabel(), plt.title()`** – подписи.
- **`plt.grid(True)`** – сетка.
- **`plt.legend()`** – легенда (если указана метка в plot).
- **`plt.xlim(), plt.ylim()`** – пределы осей.
- **`plt.show()`** – отображение.

Пример:
```python
plt.plot(t, x, label='сигнал')
plt.stem(t[::20], x[::20], linefmt='r-', markerfmt='ro', basefmt=' ')  # каждый 20-й отсчёт
plt.xlabel('Время (с)')
plt.ylabel('Амплитуда')
plt.title('Синусоида 5 Гц')
plt.legend()
plt.grid(True)
plt.show()
```

## 5. Спектральный анализ с помощью БПФ (numpy.fft)

### Вычисление спектра
```python
N = len(x)                     # количество отсчётов
X = np.fft.fft(x)              # комплексный спектр
freq = np.fft.fftfreq(N, d=1/fs)  # частоты (в Гц) для всех отсчётов
```

### Амплитудный спектр (для положительных частот)
```python
half = N // 2
freq_pos = freq[:half]
amplitude = np.abs(X[:half]) / half   # нормировка
```

### Фазовый спектр
```python
phase = np.angle(X[:half])      # фаза в радианах
```

### Спектр через scipy (альтернатива)
```python
from scipy.fftpack import fft, fftfreq   # аналогично numpy
```

## 6. Интерактивные виджеты (ipywidgets)

Базовый приём – декоратор `@interact` или функция `interact`.

### Пример интерактивного графика для демонстрации алиасинга
```python
import ipywidgets as widgets

# Функция, которая строит график по значениям амплитуды и частоты
def plot_sine(amplitude, frequency):
    x = np.linspace(0, 2 * np.pi, 200)
    y = amplitude * np.sin(frequency * x)
    
    plt.figure(figsize=(8, 4))
    plt.plot(x, y, linewidth=2)
    plt.title(f'Синусоида: амплитуда = {amplitude:.2f}, частота = {frequency:.2f}')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.grid(True)
    plt.ylim(-2, 2)   # фиксируем пределы по Y
    plt.show()

# Создаём два FloatSlider явно
amp_slider = widgets.FloatSlider(value=1.0, min=0.1, max=2.0, step=0.1, description='Амплитуда:')
freq_slider = widgets.FloatSlider(value=1.0, min=0.5, max=5.0, step=0.1, description='Частота:')

# Используем interact, передавая слайдеры как аргументы функции
widgets.interact(plot_sine, amplitude=amp_slider, frequency=freq_slider)
```

### Другие полезные виджеты
- `IntSlider`, `Dropdown`, `Checkbox` и т.д.

## 7. Полезные советы

- Для нормировки амплитудного спектра делите модуль БПФ на половину длины (`N//2`), если берёте только положительные частоты.
- Используйте `.copy()` если нужно сохранить исходные данные перед изменением.
- В Colab можно сохранять результаты на свой Google Drive через `drive.mount('/content/drive')`.